# Study: Max Price Range Ratio — 1440-sample Rolling Window

For each asset in `payamdavaee/candles`, this notebook:

1. Reads all monthly Parquet files (sorted by `ts`) via **DuckDB** directly from HuggingFace.
2. Applies a **1440-sample rolling window** (preceding + current row) over `vwap`.
3. Computes for every complete window:
   - `max_up   = |( window_max − current_vwap ) / current_vwap|`
   - `max_down = |( window_min − current_vwap ) / current_vwap|`
   - `max_range = max( max_up, max_down )`
4. Reports the **deciles** (D10 … D90) of the `max_range` series per asset.

In [ ]:
import io
import requests
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import HfApi
import duckdb
import pandas as pd
from IPython.display import display

REPO_ID = "payamdavaee/candles"
ASSETS  = ["btcusdt", "ethusdt", "trumpusdt", "vineusdt", "adausdt", "xrpusdt", "dogeusdt"]
WINDOW  = 1440

con = duckdb.connect()   # no httpfs needed — data is fetched via requests

api       = HfApi()
all_files = sorted(api.list_repo_files(REPO_ID, repo_type="dataset"))
print(f"Total files in dataset: {len(all_files)}")

In [ ]:
def get_parquet_urls(asset: str) -> list[str]:
    prefix = f"{asset}/"
    return [
        f"https://huggingface.co/datasets/{REPO_ID}/resolve/main/{f}"
        for f in all_files
        if f.startswith(prefix) and f.endswith(".parquet")
    ]


def load_asset_parquet(urls: list[str]) -> pa.Table:
    """
    Download parquet files over HTTP and return a PyArrow table with only
    the ts and vwap columns.
    """
    tables = []
    for url in urls:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        tables.append(pq.read_table(io.BytesIO(resp.content), columns=["ts", "vwap"]))
    return pa.concat_tables(tables)


def analyze_asset(asset: str, window: int = WINDOW) -> pd.DataFrame | None:
    """
    Register the asset's parquet data as an in-memory DuckDB table, run the
    rolling-window analysis, and return deciles (D10…D90 + MAX) of max_range.
    """
    urls = get_parquet_urls(asset)
    if not urls:
        print("  No Parquet files found — skipping.")
        return None

    con.register("_candles", load_asset_parquet(urls))
    try:
        query = f"""
        WITH
        windowed AS (
            SELECT
                vwap,
                MAX(vwap) OVER (ORDER BY ts ROWS BETWEEN {window - 1} PRECEDING AND CURRENT ROW) AS w_max,
                MIN(vwap) OVER (ORDER BY ts ROWS BETWEEN {window - 1} PRECEDING AND CURRENT ROW) AS w_min,
                ROW_NUMBER() OVER (ORDER BY ts) AS rn
            FROM _candles
        ),
        ranges AS (
            SELECT
                GREATEST(
                    ABS((w_max - vwap) / vwap),
                    ABS((w_min - vwap) / vwap)
                ) AS max_range
            FROM windowed
            WHERE rn >= {window}
        )
        SELECT
            unnest([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])                                           AS percentile,
            unnest(quantile_cont(max_range, [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]))      AS max_range
        FROM ranges
        """
        df = con.execute(query).df()
    finally:
        con.unregister("_candles")

    df.index = df["percentile"].apply(lambda p: "MAX" if p == 100 else f"D{p:02d}")
    df.index.name = "decile"
    return df[["max_range"]]

In [ ]:
def get_parquet_urls(asset: str) -> list[str]:
    """Return sorted HTTPS URLs for every Parquet file belonging to *asset*."""
    prefix = f"{asset}/"
    return [
        f"https://huggingface.co/datasets/{REPO_ID}/resolve/main/{f}"
        for f in all_files
        if f.startswith(prefix) and f.endswith(".parquet")
    ]


def analyze_asset(asset: str, window: int = WINDOW) -> pd.DataFrame | None:
    """
    Run the full rolling-window analysis for one asset entirely inside DuckDB.
    Returns a DataFrame with deciles (D10…D90) plus MAX of max_range, or None if no data.
    """
    urls = get_parquet_urls(asset)
    if not urls:
        print(f"  No Parquet files found — skipping.")
        return None

    # Build the quoted, comma-separated list for DuckDB's read_parquet([...])
    files_expr = ", ".join(f"'{u}'" for u in urls)

    query = f"""
    WITH

    -- 1. Load only ts + vwap, sort chronologically
    raw AS (
        SELECT ts, vwap
        FROM read_parquet([{files_expr}])
        ORDER BY ts
    ),

    -- 2. Rolling window: 1439 preceding rows + current = 1440 samples
    --    ROW_NUMBER() lets us filter out incomplete leading windows.
    windowed AS (
        SELECT
            vwap,
            MAX(vwap) OVER (ORDER BY ts ROWS BETWEEN {window - 1} PRECEDING AND CURRENT ROW) AS w_max,
            MIN(vwap) OVER (ORDER BY ts ROWS BETWEEN {window - 1} PRECEDING AND CURRENT ROW) AS w_min,
            ROW_NUMBER() OVER (ORDER BY ts) AS rn
        FROM raw
    ),

    -- 3. max_range for complete windows only (rn >= window)
    ranges AS (
        SELECT
            GREATEST(
                ABS((w_max - vwap) / vwap),
                ABS((w_min - vwap) / vwap)
            ) AS max_range
        FROM windowed
        WHERE rn >= {window}
    )

    -- 4. Deciles + maximum: unnest two parallel arrays to get one row per percentile
    SELECT
        unnest([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])                                           AS percentile,
        unnest(quantile_cont(max_range, [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]))      AS max_range
    FROM ranges
    """

    df = con.execute(query).df()
    df.index = df["percentile"].apply(lambda p: "MAX" if p == 100 else f"D{p:02d}")
    df.index.name = "decile"
    return df[["max_range"]]

In [ ]:
pd.options.display.float_format = "{:.4%}".format

for asset in ASSETS:
    urls = get_parquet_urls(asset)
    print(f"\n{'═' * 52}")
    print(f"  {asset.upper()}   ({len(urls)} monthly file(s))")
    print(f"{'═' * 52}")

    result = analyze_asset(asset)
    if result is not None:
        display(result)